# === Install Necessary Libraries ===

In [2]:
!pip install lightning timm torch-xla
!pip install --upgrade pip

# === Imports ===

# === Verify TPU ===

In [3]:
import os
import pandas as pd
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import lightning.pytorch as L
from lightning.pytorch import LightningModule, Trainer, LightningDataModule
import torch_xla.core.xla_model as xm
from sklearn.model_selection import train_test_split
from tqdm import tqdm

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/site-packages/torch_xla/__init__.py:253: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [4]:
try:
    device = xm.xla_device()
    print("✅ TPU device initialized:", device)
except Exception as e:
    print("🚫 Error initializing TPU:", e)
    device = torch.device("cpu")

E0000 00:00:1760515660.016533    2900 common_lib.cc:621] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:239


✅ TPU device initialized: xla:0


# === Data Paths ===

In [5]:
train_df = pd.read_csv('/kaggle/input/ai-vs-human-generated-dataset/train.csv')
test_df  = pd.read_csv('/kaggle/input/ai-vs-human-generated-dataset/test.csv')
img_dir  = '/kaggle/input/ai-vs-human-generated-dataset/'

# --- Create Pair IDs and Split Train/Validation (Pair-Aware) ---
train_df['pair_id'] = train_df.index // 2
unique_pairs = train_df['pair_id'].unique()

In [6]:
train_df

,Unnamed: 0,file_name,label,pair_id
0,0,train_data/a6dcb93f596a43249135678dfcfc17ea.jpg,1,0
1,1,train_data/041be3153810433ab146bc97d5af505c.jpg,0,0
2,2,train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg,1,1
3,3,train_data/8542fe161d9147be8e835e50c0de39cd.jpg,0,1
4,4,train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg,1,2
...,...,...,...,...
79945,79945,train_data/9283b107f6274279b6f15bbe77c523aa.jpg,0,39972
79946,79946,train_data/4c6b17fe6dd743428a45773135a10508.jpg,1,39973
79947,79947,train_data/1ccbf96d04e342fd9f629ad55466b29e.jpg,0,39973
79948,79948,train_data/ff960b55f296445abb3c5f304b52e104.jpg,1,39974


In [7]:
train_pairs, val_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42, shuffle=True)
train_df_split = train_df[train_df['pair_id'].isin(train_pairs)].reset_index(drop=True)
val_df_split   = train_df[train_df['pair_id'].isin(val_pairs)].reset_index(drop=True)
print("Training samples:", len(train_df_split), "Validation samples:", len(val_df_split))

Training samples: 63960 Validation samples: 15990


In [8]:
train_df_split = train_df_split.drop(columns=['pair_id'])
val_df_split = val_df_split.drop(columns=['pair_id'])

In [9]:
train_df_split = train_df_split[['file_name','label']]
val_df_split = val_df_split[['file_name','label']]

In [10]:
preds = pd.read_csv("/kaggle/input/ai-vs-human-post-processing/submission1.csv")

In [11]:
preds.rename(columns={'id': 'file_name'}, inplace=True)

In [12]:
train_df_split = pd.concat([train_df_split, preds], axis=0)
train_df_split

,file_name,label
0,train_data/a6dcb93f596a43249135678dfcfc17ea.jpg,1.0
1,train_data/041be3153810433ab146bc97d5af505c.jpg,0.0
2,train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg,1.0
3,train_data/25ea852f30594bc5915eb929682af429.jpg,0.0
4,train_data/e67085fb6d814cbabe08f978c738f3f7.jpg,1.0
...,...,...
5535,test_data_v2/483412064ff74d9d9472d606b65976d9.jpg,1.0
5536,test_data_v2/c0b49ba4081a4197b422dac7c15aea7f.jpg,0.0
5537,test_data_v2/01454aaedec140c0a3ca1f48028c41cf.jpg,0.0
5538,test_data_v2/e9adfea8b67e4791968c4c2bdd8ec343.jpg,1.0


# === Data Transforms ===

In [13]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# TTA transforms (a few variants)
tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
]

# === Custom Dataset Class ===

In [14]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        if self.is_test:
            img_path = os.path.join(self.img_dir, self.dataframe.iloc[idx]['id'])
        else:
            img_path = os.path.join(self.img_dir, self.dataframe.iloc[idx]['file_name'])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, self.dataframe.iloc[idx]['id']
        else:
            label = self.dataframe.iloc[idx]['label']
            return image, label

# === Lightning DataModule with Train/Val Split ===

In [15]:
class ImageDataModule(LightningDataModule):
    def __init__(self, train_df, val_df, test_df, img_dir, batch_size=64):
        super().__init__()
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.img_dir = img_dir
        self.batch_size = batch_size

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            self.train_dataset = CustomDataset(self.train_df, self.img_dir, transform=train_transform)
            self.val_dataset   = CustomDataset(self.val_df, self.img_dir, transform=val_transform)
        if stage == 'test' or stage is None:
            self.test_dataset = CustomDataset(self.test_df, self.img_dir, transform=val_transform, is_test=True)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

# === SqueezeNet Model ===

In [16]:
class SqueezeNetClassifier(LightningModule):
    def __init__(self, num_classes=2, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = models.squeezenet1_1(pretrained=True)
        self.model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1, 1))
        self.loss_fn = nn.CrossEntropyLoss()
        
        # Lists to store metrics
        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.loss_fn(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()

        # Log metrics for plotting later
        self.train_losses.append(loss.item())
        self.train_accs.append(acc.item())

        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.loss_fn(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()

        self.val_losses.append(loss.item())
        self.val_accs.append(acc.item())

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        return optimizer


# === Initialize DataModule ===

In [17]:
datamodule = ImageDataModule(train_df_split, val_df_split, test_df, img_dir, batch_size=64)

# === Initialize Models ===

In [18]:
model_squeezenet   = SqueezeNetClassifier(num_classes=2)

/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_1_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_1_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# === Train Each Model Separately Using Its Own Trainer Instance ===

In [29]:
# Train SqueezeNet for 3 epochs
trainer_squeezenet = Trainer(
    max_epochs=2,
    accelerator='tpu',
    devices=1,
    precision="bf16-true",
    accumulate_grad_batches=4
)
trainer_squeeze = trainer_squeezenet.fit(model_squeezenet, datamodule=datamodule)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: True, using: 1 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: True, using: 1 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: 
  | Name    | Type             | Params | Mode
----------------------------------------------------
0 | model   | SqueezeNet       | 723 K  | eval
1 | loss_fn | CrossE

/usr/local/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:527: Found 70 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 0: 100%|██████████| 1086/1086 [04:56<00:00,  3.66it/s, v_num=2, train_loss=0.375, train_acc=0.887]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 1086/1086 [05:48<00:00,  3.11it/s, v_num=2, train_loss=0.169, train_acc=0.953, val_loss=0.241, val_acc=0.977] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 1086/1086 [06:21<00:00,  2.85it/s, v_num=2, train_loss=0.169, train_acc=0.953, val_loss=0.232, val_acc=0.977]

INFO: `Trainer.fit` stopped: `max_epochs=2` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 1086/1086 [06:21<00:00,  2.85it/s, v_num=2, train_loss=0.169, train_acc=0.953, val_loss=0.232, val_acc=0.977]


In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Number of epochs
num_epochs = trainer_squeezenet.max_epochs

# Get number of batches
train_batches = len(datamodule.train_dataloader())
val_batches = len(datamodule.val_dataloader())

# Compute epoch-wise average loss
train_loss_epoch = [np.mean(model_squeezenet.train_losses[i*train_batches:(i+1)*train_batches])
                    for i in range(num_epochs)]
val_loss_epoch = [np.mean(model_squeezenet.val_losses[i*val_batches:(i+1)*val_batches])
                  for i in range(num_epochs)]

# Compute epoch-wise average accuracy
train_acc_epoch = [np.mean(model_squeezenet.train_accs[i*train_batches:(i+1)*train_batches])
                   for i in range(num_epochs)]
val_acc_epoch = [np.mean(model_squeezenet.val_accs[i*val_batches:(i+1)*val_batches])
                 for i in range(num_epochs)]

# Plot Loss
plt.figure(figsize=(10,5))
plt.plot(train_loss_epoch, label='Train Loss')
plt.plot(val_loss_epoch, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.show()

# Plot Accuracy
plt.figure(figsize=(10,5))
plt.plot(train_acc_epoch, label='Train Accuracy')
plt.plot(val_acc_epoch, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.show()


NameError: name 'trainer_squeezenet' is not defined

# === TTA Inference Functions ===

In [22]:
def tta_inference_probas(model, dataloader, tta_transforms, device):
    model.eval()
    all_probs = {}
    for batch in tqdm(dataloader, desc="TTA Inference"):
        images, img_ids = batch
        batch_probs = []
        for t in tta_transforms:
            # Convert each tensor image to PIL, apply the TTA transform, then convert back to tensor
            images_tta = torch.stack([t(transforms.ToPILImage()(img.cpu())) for img in images])
            images_tta = images_tta.to(device)
            with torch.no_grad():
                outputs = model(images_tta)
                batch_probs.append(F.softmax(outputs, dim=1))
        # Average predictions across all TTA variants
        avg_probs = torch.stack(batch_probs).mean(dim=0)
        for id_, prob in zip(img_ids, avg_probs.cpu()):
            all_probs[id_] = prob  # Save the probability vector for each image
        xm.mark_step()  # Ensure TPU synchronization
    return all_probs

def ensemble_tta_inference(models, dataloader, tta_transforms, device):
    all_model_probs = []
    for model in models:
        model = model.to(device)
        probs = tta_inference_probas(model, dataloader, tta_transforms, device)
        all_model_probs.append(probs)
    
    ensemble_probs = {}
    # Assuming all models predict on the same set of image ids,
    # average the softmax probability vectors across models.
    for img_id in all_model_probs[0].keys():
        avg_prob = sum(model_probs[img_id] for model_probs in all_model_probs) / len(all_model_probs)
        ensemble_probs[img_id] = avg_prob
    return ensemble_probs

# === Prepare Test DataLoader ===

In [23]:
datamodule.setup(stage='test')
test_loader = datamodule.test_dataloader()

# === Run Ensemble TTA Inference ===

In [24]:
model = model_squeezenet.to(device)
probs = tta_inference_probas(model, test_loader, tta_transforms, device)

TTA Inference: 100%|██████████| 87/87 [02:44<00:00,  1.89s/it]


# === Generate Final Predictions from the Ensemble ===

In [25]:
final_ensemble_preds = {img_id: torch.argmax(prob).item() for img_id, prob in probs.items()}

# === Prepare Submission DataFrame (Using test_df ordering) ===

In [26]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'label': test_df['id'].map(final_ensemble_preds)
})
submission.to_csv('submission.csv', index=False)
print("Ensemble submission file 'submission.csv' created successfully!")
print(submission.head())

Ensemble submission file 'submission.csv' created successfully!
                                                  id  label
0  test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg      1
1  test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg      1
2  test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg      1
3  test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg      1
4  test_data_v2/a16495c578b7494683805484ca27cf9f.jpg      1


In [28]:
submission['label'].value_counts()

label
1    5189
0     351
Name: count, dtype: int64